In [0]:
# =============================================================
# aggregator.py
# Layer     : Gold
# Purpose   : Aggregation logic only.
#             All groupBy, agg, sum, avg, count live here.
#             Always aggregates BEFORE joining to dimensions.
#             (Rule: never join a raw fact to a dimension —
#              aggregate first, then enrich.)
#
# STRICT RULE: No spark.read. No spark.write. No Delta.
#              No withColumn for audit — that belongs in gold_writer.
#              If you see DeltaTable here → wrong file.
# =============================================================

from pyspark.sql.functions import (
    col, sum, avg, max, min, count,
    when, lit, coalesce, round as spark_round,
    date_trunc, to_date
)


# ══════════════════════════════════════════════════════════════
# PET OWNER AGGREGATIONS
# Grain: pet_id × date
# ══════════════════════════════════════════════════════════════

def agg_activity_daily(activity_df):
    """
    Daily activity summary per pet.
    Grain: pet_id × date
    """
    return (
        activity_df
        .withColumn("date", to_date(col("snapshot_timestamp")))
        .groupBy("pet_id", "date")
        .agg(
            sum("steps").alias("total_steps"),
            sum("distance_km").alias("total_distance_km"),
            sum("active_minutes").alias("total_active_minutes")
        )
    )


def agg_health_daily(health_df):
    """
    Daily health vitals summary per pet.
    Grain: pet_id × date
    Includes health alert flag: 1 if any hour had bad vitals.
    """
    return (
        health_df
        .withColumn("date", to_date(col("snapshot_timestamp")))
        .groupBy("pet_id", "date")
        .agg(
            spark_round(avg("heart_rate"),       2).alias("avg_heart_rate"),
            spark_round(avg("body_temperature"), 2).alias("avg_body_temperature"),
            max("heart_rate").alias("max_heart_rate"),
            min("heart_rate").alias("min_heart_rate"),
            # Health alert: any hour with heart_rate=0 or body_temp out of range
            sum(
                when(
                    (col("heart_rate").isNull()) |
                    (col("body_temperature") > 41.5),
                    1
                ).otherwise(0)
            ).alias("health_alert_count")
        )
        .withColumn("has_health_alert",
            when(col("health_alert_count") > 0, True).otherwise(False))
    )


def agg_feeding_daily(feeding_df):
    """
    Daily feeding summary per pet.
    Grain: pet_id × date
    """
    return (
        feeding_df
        .withColumn("date", to_date(col("event_timestamp")))
        .groupBy("pet_id", "date")
        .agg(
            spark_round(sum("food_dispensed_grams"), 2).alias("total_food_grams"),
            count("*").alias("total_feed_sessions")
        )
    )


def agg_hydration_daily(hydration_df):
    """
    Daily hydration summary per pet.
    Grain: pet_id × date
    """
    return (
        hydration_df
        .withColumn("date", to_date(col("event_timestamp")))
        .groupBy("pet_id", "date")
        .agg(
            spark_round(sum("water_consumed_ml"), 2).alias("total_water_ml"),
            count("*").alias("total_drink_sessions")
        )
    )


def combine_pet_daily(agg_act, agg_hlth, agg_feed, agg_hyd):
    """
    FULL OUTER JOIN all four daily fact aggregates.

    Rule 1 — Full-Join Date Trap:
    A pet may have activity data but no hydration on the same day
    (e.g. the pet doesn't have a smart fountain). An inner join
    would silently drop entire days. full_outer keeps all days
    from all sources.

    All aggregates are at (pet_id, date) grain — safe to join.
    """
    return (
        agg_act
        .join(agg_hlth, ["pet_id", "date"], "full_outer")
        .join(agg_feed, ["pet_id", "date"], "left")
        .join(agg_hyd,  ["pet_id", "date"], "left")
    )


# ══════════════════════════════════════════════════════════════
# MANUFACTURER AGGREGATIONS
# Grain: device_id (lifetime summary)
# ══════════════════════════════════════════════════════════════

def agg_failures_by_device(failures_df):
    """
    Lifetime failure summary per device.
    Grain: device_id
    """
    return (
        failures_df
        .groupBy("device_id")
        .agg(
            count("failure_id").alias("total_failures"),
            spark_round(avg("severity_score"), 2).alias("avg_severity_score"),
            sum("downtime_minutes").alias("total_downtime_minutes"),
            # Crash-prone flag: more than 5 failures in lifetime
            when(count("failure_id") > 5, True)
            .otherwise(False).alias("is_crash_prone")
        )
    )


def agg_firmware_by_device(fw_df):
    """
    Firmware deployment success summary per device.
    Grain: device_id
    """
    return (
        fw_df
        .groupBy("device_id")
        .agg(
            count("deployment_id" if "deployment_id" in fw_df.columns else "*")
            .alias("total_deployments"),
            sum(when(col("deployment_status") == "Success", 1).otherwise(0))
            .alias("successful_deployments"),
            sum(when(col("rollback_flag") == True, 1).otherwise(0))
            .alias("total_rollbacks")
        )
        .withColumn("fw_success_rate_pct",
            spark_round(
                col("successful_deployments") / col("total_deployments") * 100,
                1
            )
        )
    )


def agg_device_health(snapshots_df):
    """
    Average device health metrics per device (lifetime).
    Grain: device_id
    """
    return (
        snapshots_df
        .groupBy("device_id")
        .agg(
            spark_round(avg("battery_pct"),     2).alias("avg_battery_pct"),
            spark_round(avg("signal_strength"), 2).alias("avg_signal_strength"),
            # Signal drop events: hours where signal = 0
            sum(when(col("signal_strength") == 0, 1).otherwise(0))
            .alias("signal_drop_count")
        )
    )


# ══════════════════════════════════════════════════════════════
# SALES AGGREGATIONS
# Grain: sale_month × sales_channel × device_category × country × state
# ══════════════════════════════════════════════════════════════

def agg_sales_summary(sales_enriched_df):
    """
    Monthly revenue summary grouped by channel, category, geography.

    sales_enriched_df must already have:
        sale_month, sales_channel, device_category, country, state
    enriched by joining sales → devices → products → owners upstream.

    Grain: sale_month × sales_channel × device_category × country × state
    """
    return (
        sales_enriched_df
        .groupBy(
            "sale_month", "sales_channel",
            "device_category", "country", "state"
        )
        .agg(
            count("sale_id").alias("total_units_sold"),
            spark_round(sum("sale_price"),      2).alias("gross_revenue"),
            spark_round(sum("net_revenue"),     2).alias("net_revenue"),
            spark_round(sum("discount_amount"), 2).alias("total_discounts"),
            spark_round(avg("sale_price"),      2).alias("avg_sale_price"),
            spark_round(avg("discount_amount"), 2).alias("avg_discount_amount"),
            spark_round(
                sum("discount_amount") / sum("sale_price") * 100, 1
            ).alias("discount_rate_pct")
        )
    )

print("[aggregator] Loaded. Aggregation functions ready.")